# 价格合适（Price is Right）—— 第 7 周 · 训练

## 练习目标

本笔记本对应课程 **第 7 周 Day 3 / Day 4**：用 **QLoRA** 对开源因果语言模型做 **SFT（Supervised Fine-Tuning）**，让模型学会根据商品描述预测价格。

## 和本课的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 基座模型 Base Model | `meta-llama/Llama-3.2-3B` |
| 4-bit 量化 QLoRA | `BitsAndBytesConfig` + `LoraConfig` |
| 监督微调 SFT | `trl.SFTTrainer` / `SFTConfig` |
| 实验追踪 | Weights & Biases（`wandb`）+ 推送到 Hugging Face Hub |

## 怎么跑（Colab）

1. **`LITE_MODE = True`**：免费 **T4** GPU 即可（数据集与超参更小）
2. **`LITE_MODE = False`**：请用高显存付费 GPU（如 **A100**）
3. 在 Colab Secrets 里配置 `HF_TOKEN`、`WANDB_API_KEY`
4. 从上到下依次运行单元格；训练较久，注意断线后可从 checkpoint 恢复


In [ ]:
# ========== 安装依赖 + 拉取课程工具 ==========
# -q：安静安装；固定 bitsandbytes / trl 版本，避免 Colab 默认包过旧
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
# 从课程仓库下载 week7/util.py（评估辅助函数），保存为本地 util.py
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py


In [ ]:
# ========== 导入：训练管线要用的库 ==========

# 标准库 os：读环境变量（如 WANDB_*）
import os
# 正则 re：后续工具/解析可能用到
import re
# math：数值计算辅助
import math
# tqdm：进度条，长循环时显示进度
from tqdm import tqdm
# Colab userdata：从左侧 Secrets 读取 HF_TOKEN / WANDB_API_KEY
from google.colab import userdata
# Hugging Face Hub 登录：才能下载门禁模型、push_to_hub
from huggingface_hub import login
# PyTorch：张量与 GPU
import torch
# transformers：模型/分词器生态
import transformers
# AutoModelForCausalLM / AutoTokenizer：按模型名自动加载；TrainingArguments / BitsAndBytesConfig / set_seed
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
# datasets：从 Hub 拉取 train/val/test
from datasets import load_dataset, Dataset, DatasetDict
# wandb：训练曲线与超参记录
import wandb
# peft.LoraConfig：LoRA / QLoRA 适配器配置
from peft import LoraConfig
# trl：SFTTrainer / SFTConfig —— 监督微调入口
from trl import SFTTrainer, SFTConfig
# datetime：给本次 run 起带时间戳的名字
from datetime import datetime
# matplotlib：可选画图
import matplotlib.pyplot as plt


In [ ]:
# ========== 常量与超参数：集中配置，后面 Trainer 只读这里 ==========

# 基座模型（Base Model）：Llama 3.2 3B，门禁模型需先在 HF 申请访问
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# wandb / 本地输出用的项目名
PROJECT_NAME = "price"
# 你的 Hugging Face 用户名：推送微调权重时用（改成自己的）
HF_USER = "denis-mutuma" # your HF name here!

# True=轻量模式（T4）；False=全量（建议 A100）
LITE_MODE = True

# 数据集作者
DATA_USER = "ed-donner"
# 按 LITE_MODE 在 lite / full 两套 prompt 数据集间切换
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# 本次运行名：时间戳，便于区分多次实验
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
# lite 模式在名字后加后缀，Hub / wandb 一眼可辨
if LITE_MODE:
  RUN_NAME += "-lite"
# 本地 output_dir 与部分 push 命名
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
# Hub 上的完整 repo id：用户名/项目-时间戳
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# ----- 超参数 · 总体（batch / epoch / 序列长度）-----

# lite 只跑 1 个 epoch；全量 3 个
EPOCHS = 1 if LITE_MODE else 3
# 每卡 batch；全量可用更大 batch（显存够时）
BATCH_SIZE = 32 if LITE_MODE else 256
# 截断/填充到的最大 token 长度
MAX_SEQUENCE_LENGTH = 256
# 梯度累积步数：等效放大 batch 而不占更多显存
GRADIENT_ACCUMULATION_STEPS = 1

# ----- 超参数 · QLoRA -----

# True=4-bit NF4；False 则走 8-bit 分支（见后面 BitsAndBytesConfig）
QUANT_4_BIT = True
# LoRA 秩 r：越大表达力越强、参数也越多
LORA_R = 32 if LITE_MODE else 256
# 常见经验：alpha ≈ 2 * r
LORA_ALPHA = LORA_R * 2
# 注意力投影层名字（要插入 LoRA 的模块）
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
# MLP 投影层；全量模式会一并微调
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
# lite 只改注意力；全量 = 注意力 + MLP
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
# LoRA dropout，减轻过拟合
LORA_DROPOUT = 0.1

# ----- 超参数 · 优化器与学习率 -----

LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.01
# 余弦退火学习率调度
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
# paged_adamw：bitsandbytes 分页优化器，省显存
OPTIMIZER = "paged_adamw_32bit"

# 读 GPU 算力主版本号：Ampere(8+) 可用 bf16
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# ----- 追踪与 checkpoint -----

# 验证集只取前 VAL_SIZE 条，加快 eval
VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200
# 是否把指标打到 wandb
LOG_TO_WANDB = True


In [ ]:
# ========== 检查当前 GPU 是否支持 bf16 ==========
# A100 等 Ampere+ 为 True；T4（算力 7.x）一般为 False，训练时会改用 fp16
# A100 GPU支持这个； T4本身没有

use_bf16


# 有关优化器的更多信息

文档：https://huggingface.co/docs/transformers/main/en/perf_train_gpu_one#optimizers

最常见的是 **Adam** 或 **AdamW**（Adam with Weight Decay）。  
Adam 通过存储过去梯度的滑动平均，收敛通常更稳；代价是优化器状态会带来**大约与模型参数同量级**的额外显存。  
本练习用的 `paged_adamw_32bit` 属于 bitsandbytes 的分页变体，在量化微调场景更省显存。


### 登录 Hugging Face 与 Weights & Biases

若还没有 Hugging Face 账号：打开 https://huggingface.co 注册，并创建 **Access Token**。

在 Colab 左侧钥匙图标（**Secrets**）中新增：

- `HF_TOKEN`：你的 Hugging Face token（下载门禁模型 + `push_to_hub`）
- `WANDB_API_KEY`：在 https://wandb.ai 创建后填入（训练曲线记录）

未配置密钥时，后面的 `userdata.get(...)` / `login` 会失败。


In [ ]:
# ========== 登录 Hugging Face Hub ==========

# 从 Colab Secrets 取出 HF_TOKEN（不要把 token 写进笔记本正文）
hf_token = userdata.get('HF_TOKEN')
# 登录后可拉取私有/门禁模型，并把凭证写进 git credential（便于 push）
login(hf_token, add_to_git_credential=True)


In [ ]:
# ========== 登录 Weights & Biases，并关掉过重的默认日志 ==========

# 从 Secrets 读取 WANDB_API_KEY
wandb_api_key = userdata.get('WANDB_API_KEY')
# 写入进程环境，供 wandb SDK 使用
os.environ["WANDB_API_KEY"] = wandb_api_key
# 执行登录（交互/非交互均可，密钥已在环境里）
wandb.login()

# 指定默认项目名（与 PROJECT_NAME 一致）
os.environ["WANDB_PROJECT"] = PROJECT_NAME
# 不把整模自动当 artifact 上传，省流量与时间
os.environ["WANDB_LOG_MODEL"] = "false"
# 关闭参数梯度 watch，降低开销
os.environ["WANDB_WATCH"] = "false"


In [ ]:
# ========== 加载数据集并切出 train / val / test ==========

# 按 DATASET_NAME 从 Hub 拉取（含 prompt 字段，供 SFT 使用）
dataset = load_dataset(DATASET_NAME)
# 训练集：全部用于 SFT
train = dataset['train']
# 验证集：只取前 VAL_SIZE，加快中间 eval
val = dataset['val'].select(range(VAL_SIZE))
# 测试集：本笔记本训练阶段通常不直接用；留给评估本
test = dataset['test']


In [ ]:
# ========== 可选：缩小训练集（默认保持注释，逻辑不变）==========
# 若只想快速冒烟测试，可取消下一行注释，把训练集约到 10,000 条
# 注意：标识符须保持英文 train / range（勿改成中文）

# train = train.select(range(10000))


In [ ]:
# ========== 启动 wandb run（若开启日志）==========
# project / name 与前面常量一致，便于在网页上对照超参
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)


## 现在加载分词器（Tokenizer）和基座模型

模型会被 **量化（Quantization）**：把权重精度降到 **4-bit（NF4）**（或 8-bit），从而在消费级 GPU 上塞进 3B 级模型，并为 QLoRA 微调做准备。


In [ ]:
# ========== 按 QUANT_4_BIT 选择 BitsAndBytes 量化配置 ==========

if QUANT_4_BIT:
  # 4-bit NF4 + double quant：QLoRA 常见组合
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # 对量化常数再量化，进一步省显存
    bnb_4bit_use_double_quant=True,
    # 计算 dtype：bf16 优先，否则 fp16
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  # 8-bit 量化分支（当 QUANT_4_BIT=False）
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )


In [ ]:
# ========== 加载分词器与量化后的因果语言模型 ==========

# trust_remote_code=True：允许模型仓库里的自定义代码（若有）
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# 因果 LM 常无 pad_token：用 eos 充当，避免 batch 报错
tokenizer.pad_token = tokenizer.eos_token
# 右侧 padding：生成/训练时更常见
tokenizer.padding_side = "right"

# device_map="auto"：accelerate 自动把层放到可用 GPU
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# 生成配置里的 pad_token_id 与 tokenizer 对齐
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# 打印显存占用（MB），确认量化是否生效
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")


# 配置训练：两个关键对象

接下来要创建：

1. **`LoraConfig`**：LoRA 秩、alpha、dropout、要注入的 `target_modules`
2. **`SFTConfig`**：epoch、batch、学习率、保存/评估步数、是否 `push_to_hub` 等整体训练参数

两者交给 `SFTTrainer` 后即可开训。


In [ ]:
# ========== LoRA / QLoRA 适配器超参 ==========

lora_parameters = LoraConfig(
    # 缩放系数 alpha（与 r 联动）
    lora_alpha=LORA_ALPHA,
    # LoRA 层 dropout
    lora_dropout=LORA_DROPOUT,
    # 低秩维度 r
    r=LORA_R,
    # 不对 bias 做 LoRA
    bias="none",
    # 因果语言建模任务类型
    task_type="CAUSAL_LM",
    # 插入 LoRA 的模块名列表（注意力 ± MLP）
    target_modules=TARGET_MODULES,
)


In [ ]:
# ========== SFT 训练参数（SFTConfig）==========

train_parameters = SFTConfig(
    # checkpoint / 日志输出目录
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    # eval 用较小 batch，省显存
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    # 最多保留最近 10 个 checkpoint，防占满磁盘
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    # T4 等：fp16；A100+：bf16（互斥）
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    # -1 表示不按 max_steps 截断，按 epoch 跑完
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    # 相近长度样本组 batch，减少 padding 浪费
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    # 有 wandb 就上报；否则 report_to=None
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    # SFT 最大序列长度
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    # 每次 save 都尝试同步到 Hub
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    # 私有仓库，避免未 scrub 的权重公开
    hub_private_repo=True,
    eval_strategy="steps",
    # 与 save 同频做验证
    eval_steps=SAVE_STEPS
)


# 现在 —— 创建训练器（`SFTTrainer`）


In [ ]:
# ========== 组装 SFTTrainer：基座 + 数据 + LoRA + 训练参数 ==========
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    # PEFT / LoRA 配置
    peft_config=lora_parameters,
    args=train_parameters
)


## 下一个单元格：开始微调！

训练会跑一段时间，并且大约每 `SAVE_STEPS` 步把 checkpoint 推到 Hugging Face Hub。

### Colab 可能中途断开

- **免费计划**：资源紧张时可能被回收
- **付费计划**：单次会话最长约 24 小时，也不保证不中断

若中断，可参考作者提供的「从上次保存恢复」Colab：

https://colab.research.google.com/drive/1qGTDVIas_Vwoby4UVi2vwsU0tHXy8OMO#scrollTo=R_O04fKxMMT-

该示例已保留最终运行输出。恢复时的关键点：加载微调模型时设置 **`is_trainable=True`**，才能继续训练。

准备好后，运行下一格开始 `train()`。


In [ ]:
# ========== 开训，并把最终适配器推到 Hub ==========
# 阻塞直到训练结束（或被 Colab 中断）
fine_tuning.train()

# 再显式 push 一份到以 PROJECT_RUN_NAME 命名的私有仓库
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")


In [ ]:
# ========== 收尾：关闭 wandb run，刷完剩余日志 ==========
if LOG_TO_WANDB:
  wandb.finish()
